In [1]:
# -*- coding: utf-8 -*-
"""
matroid_vs_nonmatroid_explained.py

マトロイド vs 非マトロイド の可視化完全版

機能
1. マトロイド側:
   - グラフィックマトロイド（閉路を含まない辺集合）
   - 独立集合サイズ表示
   - matroid rank 表示
   - cycle があれば赤表示

2. 非マトロイド側:
   - 「A群から高々1本、B群から高々2本」の独立系
   - 遺伝性はあるが、交換公理が壊れる例
   - 交換公理の反例 A, B を自動探索
   - B-A のうち A に追加できない辺を赤表示

必要:
    pip install networkx matplotlib
"""

import itertools
import random
from typing import List, Tuple, Set, FrozenSet, Callable

import networkx as nx
import matplotlib.pyplot as plt


# =========================================================
# 基本設定
# =========================================================

SEED = 42
random.seed(SEED)

NODES = [0, 1, 2, 3, 4]

# 候補辺
CANDIDATE_EDGES = [
    (0, 1), (0, 2), (0, 3), (0, 4),
    (1, 2), (1, 3), (1, 4),
    (2, 3), (2, 4),
    (3, 4),
]

# 非マトロイド用の2群
# A群: 高々1本
# B群: 高々2本
GROUP_A = {tuple(sorted(e)) for e in [(0, 1), (0, 2), (0, 3)]}
GROUP_B = {tuple(sorted(e)) for e in [(1, 2), (1, 3), (2, 3)]}
# それ以外の辺は自由

# 描画位置
POS = {
    0: (0.0, 1.0),
    1: (1.0, 1.8),
    2: (2.0, 1.0),
    3: (1.0, 0.2),
    4: (3.0, 1.0),
}


# =========================================================
# 基本関数
# =========================================================

def norm_edge(e: Tuple[int, int]) -> Tuple[int, int]:
    a, b = e
    return (a, b) if a < b else (b, a)


def norm_edge_set(edge_set) -> FrozenSet[Tuple[int, int]]:
    return frozenset(norm_edge(e) for e in edge_set)


def all_subsets(edges: List[Tuple[int, int]]):
    edges = [norm_edge(e) for e in edges]
    for r in range(len(edges) + 1):
        for comb in itertools.combinations(edges, r):
            yield frozenset(comb)


def build_graph(nodes: List[int], edge_set: Set[Tuple[int, int]]) -> nx.Graph:
    g = nx.Graph()
    g.add_nodes_from(nodes)
    g.add_edges_from(edge_set)
    return g


def set_to_str(edge_set):
    return "{" + ", ".join(str(e) for e in sorted(edge_set)) + "}"


# =========================================================
# マトロイド側: グラフィックマトロイド
# =========================================================

def is_graphic_independent(nodes: List[int], edge_set: Set[Tuple[int, int]]) -> bool:
    g = build_graph(nodes, edge_set)
    return nx.is_forest(g)


def graphic_rank(nodes: List[int], candidate_edges: List[Tuple[int, int]]) -> int:
    """
    グラフィックマトロイドの rank = |V| - 連結成分数
    ただし候補辺だけで作るグラフ上の連結成分数
    """
    g = build_graph(nodes, candidate_edges)
    c = nx.number_connected_components(g)
    return len(nodes) - c


def random_graphic_independent_set(
    nodes: List[int],
    candidate_edges: List[Tuple[int, int]],
    p: float = 0.8,
) -> FrozenSet[Tuple[int, int]]:
    edges = [norm_edge(e) for e in candidate_edges]
    random.shuffle(edges)
    chosen = set()

    for e in edges:
        if random.random() > p:
            continue
        trial = set(chosen)
        trial.add(e)
        if is_graphic_independent(nodes, trial):
            chosen.add(e)

    return frozenset(chosen)


def find_cycle_edges(nodes: List[int], edge_set: Set[Tuple[int, int]]) -> Set[Tuple[int, int]]:
    """
    cycle があれば、その cycle を構成する辺を返す
    """
    g = build_graph(nodes, edge_set)
    try:
        cyc = nx.find_cycle(g, orientation="ignore")
        cyc_edges = {norm_edge((u, v)) for (u, v, *_) in cyc}
        return cyc_edges
    except nx.exception.NetworkXNoCycle:
        return set()


# =========================================================
# 非マトロイド側
# 独立条件:
# - GROUP_A から高々1本
# - GROUP_B から高々2本
#
# これは遺伝性を満たすが、交換公理が壊れる
# 例:
# A = {B群の2本}
# B = {A群の1本}
# |B| < |A| の逆向きだと見にくいので、
# 実際には全探索で反例を探す
# =========================================================

def is_nonmatroid_independent(nodes: List[int], edge_set: Set[Tuple[int, int]]) -> bool:
    edge_set = {norm_edge(e) for e in edge_set}

    count_a = len(edge_set & GROUP_A)
    count_b = len(edge_set & GROUP_B)

    if count_a > 1:
        return False
    if count_b > 2:
        return False

    return True


def nonmatroid_rank(candidate_edges: List[Tuple[int, int]]) -> int:
    """
    全探索で最大独立集合サイズを返す
    """
    best = 0
    for s in all_subsets(candidate_edges):
        if is_nonmatroid_independent(NODES, s):
            best = max(best, len(s))
    return best


def random_nonmatroid_independent_set(
    nodes: List[int],
    candidate_edges: List[Tuple[int, int]],
    p: float = 0.8,
) -> FrozenSet[Tuple[int, int]]:
    edges = [norm_edge(e) for e in candidate_edges]
    random.shuffle(edges)
    chosen = set()

    for e in edges:
        if random.random() > p:
            continue
        trial = set(chosen)
        trial.add(e)
        if is_nonmatroid_independent(nodes, trial):
            chosen.add(e)

    return frozenset(chosen)


# =========================================================
# 公理チェック
# =========================================================

def enumerate_independent_sets(
    nodes: List[int],
    candidate_edges: List[Tuple[int, int]],
    independence_fn: Callable[[List[int], Set[Tuple[int, int]]], bool],
) -> Set[FrozenSet[Tuple[int, int]]]:
    indep = set()
    for s in all_subsets(candidate_edges):
        if independence_fn(nodes, s):
            indep.add(s)
    return indep


def hereditary_property(indep_sets: Set[FrozenSet[Tuple[int, int]]]) -> bool:
    for s in indep_sets:
        for r in range(len(s) + 1):
            for sub in itertools.combinations(s, r):
                if frozenset(sub) not in indep_sets:
                    return False
    return True


def exchange_property(indep_sets: Set[FrozenSet[Tuple[int, int]]]) -> bool:
    indep_list = list(indep_sets)
    for A in indep_list:
        for B in indep_list:
            if len(A) < len(B):
                found = False
                for e in (B - A):
                    if frozenset(set(A) | {e}) in indep_sets:
                        found = True
                        break
                if not found:
                    return False
    return True


def find_exchange_counterexample(indep_sets: Set[FrozenSet[Tuple[int, int]]]):
    indep_list = list(indep_sets)
    indep_list.sort(key=lambda s: (len(s), sorted(s)))
    for A in indep_list:
        for B in indep_list:
            if len(A) < len(B):
                blockers = []
                for e in (B - A):
                    if frozenset(set(A) | {e}) not in indep_sets:
                        blockers.append(e)
                if len(blockers) == len(B - A) and len(blockers) > 0:
                    return A, B, blockers
    return None, None, None


# =========================================================
# 補助説明
# =========================================================

def explain_graphic_set(nodes, edge_set):
    cycle_edges = find_cycle_edges(nodes, edge_set)
    if len(cycle_edges) == 0:
        return "理由: 閉路なし → forest → グラフィックマトロイドの独立集合"
    return "理由: cycle があるので独立ではない"


def explain_nonmatroid_set(edge_set):
    count_a = len(set(edge_set) & GROUP_A)
    count_b = len(set(edge_set) & GROUP_B)
    return f"理由: A群 {count_a}/1本, B群 {count_b}/2本"


# =========================================================
# 描画
# =========================================================

def draw_base_groups(ax):
    # A群を薄い青破線, B群を薄い緑破線で背景表示
    for e in GROUP_A:
        x1, y1 = POS[e[0]]
        x2, y2 = POS[e[1]]
        ax.plot([x1, x2], [y1, y2], linestyle="--", linewidth=1, alpha=0.25)

    for e in GROUP_B:
        x1, y1 = POS[e[0]]
        x2, y2 = POS[e[1]]
        ax.plot([x1, x2], [y1, y2], linestyle=":", linewidth=1.5, alpha=0.25)


def draw_graph_with_annotations(
    ax,
    nodes: List[int],
    edge_set: Set[Tuple[int, int]],
    title: str,
    subtitle: str,
    rank_value: int,
    red_edges: Set[Tuple[int, int]] = None,
    orange_edges: Set[Tuple[int, int]] = None,
):
    if red_edges is None:
        red_edges = set()
    if orange_edges is None:
        orange_edges = set()

    g = build_graph(nodes, edge_set)

    draw_base_groups(ax)

    # ノード
    nx.draw_networkx_nodes(g, POS, ax=ax, node_size=700)
    nx.draw_networkx_labels(g, POS, ax=ax, font_size=11)

    # 通常辺
    normal_edges = [e for e in g.edges() if norm_edge(e) not in red_edges and norm_edge(e) not in orange_edges]
    if normal_edges:
        nx.draw_networkx_edges(g, POS, ax=ax, edgelist=normal_edges, width=2)

    # オレンジ辺
    orange_list = [e for e in g.edges() if norm_edge(e) in orange_edges]
    if orange_list:
        nx.draw_networkx_edges(g, POS, ax=ax, edgelist=orange_list, width=3)

    # 赤辺
    red_list = [e for e in g.edges() if norm_edge(e) in red_edges]
    if red_list:
        nx.draw_networkx_edges(g, POS, ax=ax, edgelist=red_list, width=4)

    text = (
        f"{title}\n"
        f"{subtitle}\n"
        f"独立集合サイズ = {len(edge_set)}\n"
        f"rank = {rank_value}\n"
        f"{set_to_str(edge_set)}"
    )
    ax.set_title(text, fontsize=9)
    ax.axis("off")


# =========================================================
# メイン
# =========================================================

def main():
    print("=== 候補辺 ===")
    print(set_to_str(CANDIDATE_EDGES))
    print()

    print("=== 非マトロイド用の群 ===")
    print("GROUP_A (高々1本):", set_to_str(GROUP_A))
    print("GROUP_B (高々2本):", set_to_str(GROUP_B))
    print()

    # 全独立集合
    graphic_indep_sets = enumerate_independent_sets(NODES, CANDIDATE_EDGES, is_graphic_independent)
    nonmatroid_indep_sets = enumerate_independent_sets(NODES, CANDIDATE_EDGES, is_nonmatroid_independent)

    # 公理チェック
    print("=== マトロイド側 ===")
    print("空集合:", frozenset() in graphic_indep_sets)
    print("遺伝性:", hereditary_property(graphic_indep_sets))
    print("交換公理:", exchange_property(graphic_indep_sets))
    print()

    print("=== 非マトロイド側 ===")
    print("空集合:", frozenset() in nonmatroid_indep_sets)
    print("遺伝性:", hereditary_property(nonmatroid_indep_sets))
    print("交換公理:", exchange_property(nonmatroid_indep_sets))
    print()

    # rank
    r_graphic = graphic_rank(NODES, CANDIDATE_EDGES)
    r_nonmatroid = nonmatroid_rank(CANDIDATE_EDGES)

    print("グラフィックマトロイド rank =", r_graphic)
    print("非マトロイド独立系の最大独立集合サイズ =", r_nonmatroid)
    print()

    # 反例
    A, B, blockers = find_exchange_counterexample(nonmatroid_indep_sets)
    if A is None:
        raise RuntimeError("交換公理の反例が見つかりませんでした。群の設定を変更してください。")

    print("=== 交換公理の反例 ===")
    print("A =", set_to_str(A))
    print("B =", set_to_str(B))
    print("|A| =", len(A), ", |B| =", len(B))
    print("B - A =", set_to_str(B - A))
    print("A に追加できない辺 =", set_to_str(blockers))
    print()

    # サンプル生成
    graphic_samples = [random_graphic_independent_set(NODES, CANDIDATE_EDGES, p=0.85) for _ in range(3)]
    nonmatroid_samples = [random_nonmatroid_independent_set(NODES, CANDIDATE_EDGES, p=0.85) for _ in range(2)]

    # マトロイド側の一つに cycle を人工的に追加した説明用サンプル
    graphic_bad = frozenset({(0, 1), (1, 2), (0, 2)})

    # -------------------------
    # 図1: マトロイド側
    # -------------------------
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))

    for i, s in enumerate(graphic_samples):
        cycle_edges = find_cycle_edges(NODES, s)
        subtitle = explain_graphic_set(NODES, s)
        draw_graph_with_annotations(
            axes[i],
            NODES,
            s,
            title=f"Matroid sample {i+1}",
            subtitle=subtitle,
            rank_value=r_graphic,
            red_edges=cycle_edges,
        )

    # 最後は cycle 付き説明用
    bad_cycle_edges = find_cycle_edges(NODES, graphic_bad)
    draw_graph_with_annotations(
        axes[3],
        NODES,
        graphic_bad,
        title="Cycle example",
        subtitle="理由: 赤辺が cycle を作る → 独立でない",
        rank_value=r_graphic,
        red_edges=bad_cycle_edges,
    )

    plt.suptitle("グラフィックマトロイド: 閉路なしなら独立、cycle は赤表示", fontsize=14)
    plt.tight_layout()
    plt.show()

    # -------------------------
    # 図2: 非マトロイド側
    # -------------------------
    fig, axes = plt.subplots(1, 4, figsize=(22, 5))

    # 反例A
    draw_graph_with_annotations(
        axes[0],
        NODES,
        A,
        title="Non-matroid counterexample A",
        subtitle=(
            "理由: 独立だが、より大きい B から\n"
            "追加できる辺が1本もない"
        ),
        rank_value=r_nonmatroid,
        red_edges=set(),
        orange_edges=set(A & GROUP_A) | set(A & GROUP_B),
    )

    # 反例B
    draw_graph_with_annotations(
        axes[1],
        NODES,
        B,
        title="Non-matroid counterexample B",
        subtitle=(
            "理由: B-A の赤辺は A に足すと制約違反\n"
            "→ 交換公理が破れる"
        ),
        rank_value=r_nonmatroid,
        red_edges=set(blockers),
        orange_edges=(set(B) - set(blockers)),
    )

    # A + 各 blocker を試す図を1つ
    if len(blockers) > 0:
        e0 = blockers[0]
        A_plus = frozenset(set(A) | {e0})
        draw_graph_with_annotations(
            axes[2],
            NODES,
            A_plus,
            title="A + red edge",
            subtitle=(
                f"理由: 赤辺 {e0} を追加すると独立性が壊れる\n"
                f"{explain_nonmatroid_set(A_plus)}"
            ),
            rank_value=r_nonmatroid,
            red_edges={e0},
            orange_edges=(set(A_plus) - {e0}),
        )

    # ランダムサンプル
    s = nonmatroid_samples[0]
    draw_graph_with_annotations(
        axes[3],
        NODES,
        s,
        title="Random non-matroid sample",
        subtitle=explain_nonmatroid_set(s),
        rank_value=r_nonmatroid,
        orange_edges=set(s & GROUP_A) | set(s & GROUP_B),
    )

    plt.suptitle(
        "非マトロイド独立系: A群≤1本, B群≤2本。交換公理の破れた候補辺を赤表示",
        fontsize=14
    )
    plt.tight_layout()
    plt.show()

    # -------------------------
    # 図3: 反例を文章でも表示
    # -------------------------
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.axis("off")

    text = (
        "交換公理の反例\n\n"
        f"A = {set_to_str(A)}\n"
        f"B = {set_to_str(B)}\n"
        f"|A| = {len(A)}, |B| = {len(B)}\n"
        f"B - A = {set_to_str(B - A)}\n"
        f"A に追加できない辺 = {set_to_str(blockers)}\n\n"
        "意味:\n"
        "A は独立、B も独立、しかも |A| < |B| なのに、\n"
        "B-A のどの辺を A に足しても独立でなくなる。\n"
        "したがって交換公理が壊れ、マトロイドではない。"
    )
    ax.text(0.02, 0.95, text, va="top", fontsize=12)
    plt.tight_layout()
    plt.show()


if __name__ == "__main__":
    main()

=== 候補辺 ===
{(0, 1), (0, 2), (0, 3), (0, 4), (1, 2), (1, 3), (1, 4), (2, 3), (2, 4), (3, 4)}

=== 非マトロイド用の群 ===
GROUP_A (高々1本): {(0, 1), (0, 2), (0, 3)}
GROUP_B (高々2本): {(1, 2), (1, 3), (2, 3)}

=== マトロイド側 ===
空集合: True
遺伝性: True
交換公理: True

=== 非マトロイド側 ===
空集合: True
遺伝性: True
交換公理: True

グラフィックマトロイド rank = 4
非マトロイド独立系の最大独立集合サイズ = 7



RuntimeError: 交換公理の反例が見つかりませんでした。群の設定を変更してください。